In [ ]:
# Import Google Drive access library from Google Colab
# This allows the program to read files stored in your Google Drive
from google.colab import drive

# Connect (mount) Google Drive to the Colab environment
# After this step, files in Google Drive can be accessed like normal folders
drive.mount('/content/drive')


# Import OS library
# Helps the program work with file paths and directories
import os

# Import NumPy library
# Used for handling image data in numerical array format
import numpy as np

# Import glob library
# Used to search for files in a folder that match specific patterns (like *.jpg)
import glob

# Import function to load a previously trained machine learning model
from tensorflow.keras.models import load_model

# Import image processing tools
# These help load images and convert them into a format the AI model understands
from tensorflow.keras.preprocessing import image


# Define the location of the trained model file in Google Drive
# This model was created during the training phase
model_path = '/content/drive/MyDrive/ML_WSSV/ML_WSSV/WSSV/Trained_Model/efficientnetb0_model.h5'

# Define the folder that contains shrimp images for testing
# These images will be analyzed by the trained AI model
test_images_path = '/content/drive/MyDrive/ML_WSSV/WSSV/images'


# Load the trained machine learning model from the specified path
# This model already learned how to detect WSSV and Healthy shrimp
model = load_model(model_path)

# Print confirmation message showing the model was successfully loaded
print(f"✅ Model loaded from: {model_path}")


# Define a function that predicts whether a shrimp image is Healthy or WSSV infected
# It takes two inputs:
# 1. image_path → location of the shrimp image
# 2. model → the trained machine learning model
def predict_image_class(image_path, model):

    # Load the image from the given file path
    # Resize the image to 224x224 pixels because the model expects this size
    img = image.load_img(image_path, target_size=(224, 224))

    # Convert the image into a numeric array
    # Machine learning models process numbers rather than image files
    img_array = image.img_to_array(img) / 255.0

    # Normalize pixel values by dividing by 255
    # This converts pixel values from range (0–255) to (0–1)
    # Normalization improves model prediction stability

    # Expand the array dimension to simulate a batch of images
    # Even if we predict one image, the model expects a batch format
    img_array = np.expand_dims(img_array, axis=0)

    # Use the trained model to predict the probability
    # The result will be a number between 0 and 1
    prediction = model.predict(img_array, verbose=0)[0][0]

    # If prediction value is greater than 0.5 → classify as WSSV
    # Otherwise classify as Healthy shrimp
    class_label = 'WSSV' if prediction > 0.5 else 'HEALTHY'

    # Calculate prediction confidence
    # If the model predicts WSSV → use prediction value
    # If the model predicts Healthy → use inverse probability
    confidence = prediction if prediction > 0.5 else 1 - prediction

    # Return predicted label and confidence percentage
    return class_label, round(confidence * 100, 2)


# Define the image formats that the program should search for
# This allows the system to detect multiple image file types
image_extensions = ['*.jpg', '*.jpeg', '*.png']


# Create an empty list to store all image file paths
all_images = []

# Loop through each image extension type
for ext in image_extensions:

    # Search the image folder for files that match the extension
    # Example: all .jpg or .png files
    all_images.extend(glob.glob(os.path.join(test_images_path, ext)))


# Print message showing which folder is being scanned
print(f"📁 Scanning folder: {test_images_path}")

# Print how many images were found in the folder
print(f"🧾 Found {len(all_images)} image(s).")


# Loop through every image found in the folder
for img_path in all_images:

    try:
        # Call the prediction function for each image
        label, conf = predict_image_class(img_path, model)

        # Print the prediction result
        # Shows image name, predicted class, and confidence percentage
        print(f"🖼️ {os.path.basename(img_path)} → Predicted: {label} ({conf}% confidence)")

    except Exception as e:
        # If any error occurs (such as corrupted image)
        # Print an error message instead of stopping the program
        print(f"❌ Error with {os.path.basename(img_path)}: {e}")

Mounted at /content/drive


✅ Model loaded from: /content/drive/MyDrive/ML_WSSV/shrimpInfectionDetection/efficientnetb0_model.h5
📁 Scanning folder: /content/drive/MyDrive/ML_WSSV/shrimpInfectionDetection/images
🧾 Found 2 image(s).
🖼️ Screenshot 2023-11-09 at 12.56.38.png → Predicted: WSSV (63.29999923706055% confidence)
🖼️ Screenshot 2023-11-09 at 12.57.32.png → Predicted: WSSV (63.279998779296875% confidence)
